# 09 - Scalability Test: Second CNN Iteration

This notebook measures how model-only inference time changes as the number of images increases. It uses validation images only, preloads and preprocesses them once, then repeats those tensors deterministically to create larger workloads. The notebook does not train or tune the model.


## Setup

The setup matches the selected second CNN iteration. The timing test uses the same preprocessing, frozen DenseNet201 prefix, and fine-tuned DenseNet201 tail as the final test evaluation.


In [ ]:
from __future__ import annotations

from collections import OrderedDict
from pathlib import Path
import os
import random
from typing import Iterable

local_torch_home = Path.home() / ".cache" / "torch"
local_densenet_weights = local_torch_home / "hub" / "checkpoints" / "densenet201-c1103571.pth"
if local_densenet_weights.exists():
    os.environ.setdefault("TORCH_HOME", str(local_torch_home))
else:
    os.environ.setdefault("TORCH_HOME", "/tmp/mdpp_v2_notebook_cache/torch")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mdpp_v2_notebook_cache/matplotlib")

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
from torchvision.models import DenseNet201_Weights, densenet201

RANDOM_SEED = 42
PRELOAD_BATCH_SIZE = 16
INFERENCE_BATCH_SIZE = 64
IMAGE_MODALITY = "Stereo"
IMAGE_INSTRUMENT = "T"
SELECTED_SETTING = "Colour + mono + polarized grayscale"
FINETUNED_CNN_CHECKPOINT_NAME = "cnn_finetuned_densenet201_denseblock4_head.pt"

CLASS_NAMES = [
    "Cotton",
    "Natural polyamide",
    "Polyester / recycled polyester / PET",
    "Regenerated cellulosic",
    "Synthetic polyamide / nylon",
]
LABEL_TO_INDEX = {label: index for index, label in enumerate(CLASS_NAMES)}
INDEX_TO_LABEL = {index: label for label, index in LABEL_TO_INDEX.items()}
FEATURE_SOURCES = [
    {"feature_source": "C_gray", "formats": ("C",), "preprocessing": "L_to_RGB"},
    {"feature_source": "M_gray", "formats": ("M",), "preprocessing": "L_to_RGB"},
    {"feature_source": "P_gray", "formats": ("P",), "preprocessing": "L_to_RGB"},
]
SELECTED_FEATURE_SOURCES = tuple(source["feature_source"] for source in FEATURE_SOURCES)
PAIR_GROUP_ORDER = ["Same fabric", "Same class, different fabric", "Different class"]
PAIR_GROUP_LABELS = ["Same fabric", "Same class,\ndifferent fabric", "Different class"]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))


def select_torch_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    raise RuntimeError("CUDA is not available. Fix GPU visibility before running this notebook.")


DEVICE = select_torch_device()
PIN_MEMORY = DEVICE.type == "cuda"
print(f"Using device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"Torch home: {os.environ['TORCH_HOME']}")

## 1. Paths, Validation Split, and Output Files


In [ ]:
def resolve_mdpp_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "mdpp_v2" / "data_splits").exists():
            return candidate / "mdpp_v2"
        if candidate.name == "mdpp_v2" and (candidate / "data_splits").exists():
            return candidate
    raise FileNotFoundError("Could not resolve mdpp_v2 root from current working directory.")


MDPP_ROOT = resolve_mdpp_root()
DATA_SPLITS_DIR = MDPP_ROOT / "data_splits"
ARTIFACTS_DIR = MDPP_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FINETUNED_CNN_CHECKPOINT_PATH = ARTIFACTS_DIR / FINETUNED_CNN_CHECKPOINT_NAME
SCALABILITY_TIMING_PATH = ARTIFACTS_DIR / "validation_image_count_inference_timing.csv"
SCALABILITY_PLOT_PATH = ARTIFACTS_DIR / "validation_image_count_inference_timing.png"

SPLITS = ["val"]
split_sample_tables = {split: pd.read_csv(DATA_SPLITS_DIR / split / "samples.csv") for split in SPLITS}
split_manifest_tables = {split: pd.read_csv(DATA_SPLITS_DIR / split / "measurements_manifest.csv") for split in SPLITS}
if not FINETUNED_CNN_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing selected checkpoint: {FINETUNED_CNN_CHECKPOINT_PATH}")
print(f"Loaded splits: {', '.join(SPLITS)}")
print(f"Selected checkpoint: {FINETUNED_CNN_CHECKPOINT_PATH}")


## 2. Build Validation Image Table


In [ ]:
def eligible_samples_for_split(split: str) -> pd.DataFrame:
    samples = split_sample_tables[split].copy()
    eligible = samples[samples["cnn_label"].ne("EXCLUDED_CNN")].copy()
    eligible = eligible[["Sample ID", "classification_label", "cnn_label", "split"]]
    eligible = eligible.rename(columns={"Sample ID": "sample_id"})
    eligible["label_index"] = eligible["classification_label"].map(LABEL_TO_INDEX)
    if eligible["label_index"].isna().any():
        missing = sorted(eligible.loc[eligible["label_index"].isna(), "classification_label"].dropna().unique())
        raise ValueError(f"Unexpected retained CNN labels: {missing}")
    return eligible


def build_source_table(split: str, feature_source: str, formats: Iterable[str], preprocessing: str) -> pd.DataFrame:
    manifest = split_manifest_tables[split].copy()
    eligible = eligible_samples_for_split(split)
    mask = (
        manifest["modality_group"].eq("image")
        & manifest["modality"].eq(IMAGE_MODALITY)
        & manifest["instrument"].eq(IMAGE_INSTRUMENT)
        & manifest["data_format"].isin(tuple(formats))
    )
    table = manifest.loc[mask].merge(eligible, on="sample_id", how="inner", suffixes=("", "_sample"))
    table["path"] = table["final_path"].map(Path)
    table["image_path"] = table["path"]
    table["feature_source"] = feature_source
    table["preprocessing"] = preprocessing
    table["setting"] = SELECTED_SETTING
    table["source_split"] = split
    table["image_id"] = table["source_split"] + ":" + table["feature_source"] + ":" + table["filename"]
    if table["cnn_label"].eq("EXCLUDED_CNN").any():
        raise ValueError("EXCLUDED_CNN rows entered the classifier table.")
    missing_files = [path for path in table["path"] if not path.exists()]
    if missing_files:
        raise FileNotFoundError(f"Missing image files, first examples: {missing_files[:3]}")
    return table.sort_values(["sample_id", "feature_source", "replicate", "filename"]).reset_index(drop=True)


def make_feature_input_table(frame: pd.DataFrame) -> pd.DataFrame:
    original = frame.copy()
    original["augmentation_variant"] = "original"
    original["feature_input_id"] = original["image_id"] + ":original"
    return original.reset_index(drop=True)


validation_tables = []
for source_spec in FEATURE_SOURCES:
    validation_tables.append(
        build_source_table(
            split="val",
            feature_source=source_spec["feature_source"],
            formats=source_spec["formats"],
            preprocessing=source_spec["preprocessing"],
        )
    )
validation_table = (
    pd.concat(validation_tables, ignore_index=True)
    .sort_values(["sample_id", "feature_source", "replicate", "filename"])
    .reset_index(drop=True)
)
validation_feature_frame = make_feature_input_table(validation_table)

image_input_summary = pd.DataFrame([
    {
        "split": "validation",
        "rows": len(validation_table),
        "fabrics": validation_table["sample_id"].nunique(),
        "classes": validation_table["classification_label"].nunique(),
    },
])
image_input_summary


## 3. Selected Fine-Tuned DenseNet201 Model

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform: transforms.Compose):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        image = Image.open(row["image_path"]).convert("L").convert("RGB")
        image = self.transform(image)
        label = int(row["label_index"])
        return image, label, row["feature_input_id"]


weights = DenseNet201_Weights.DEFAULT
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.transforms().mean, std=weights.transforms().std),
])


def get_module(model: nn.Module, dotted_name: str) -> nn.Module:
    module = model
    for name_part in dotted_name.split("."):
        module = getattr(module, name_part)
    return module


class DenseNet201Tail(nn.Module):
    def __init__(self, denseblock4: nn.Module, norm5: nn.Module, classifier: nn.Linear):
        super().__init__()
        self.denseblock4 = denseblock4
        self.norm5 = norm5
        self.classifier = classifier

    def forward(self, prefix_features: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        features = self.denseblock4(prefix_features)
        features = self.norm5(features)
        features = F.relu(features, inplace=False)
        pooled = F.adaptive_avg_pool2d(features, (1, 1))
        embedding = torch.flatten(pooled, 1)
        logits = self.classifier(embedding)
        return logits, embedding


base_densenet = densenet201(weights=weights)
feature_dim = base_densenet.classifier.in_features
base_densenet.classifier = nn.Linear(feature_dim, len(CLASS_NAMES))

unfrozen_feature_names = {"denseblock4", "norm5"}
prefix_modules = OrderedDict(
    (name, module)
    for name, module in base_densenet.features.named_children()
    if name not in unfrozen_feature_names
)
frozen_prefix = nn.Sequential(prefix_modules).to(DEVICE).eval()
tail_model = DenseNet201Tail(
    base_densenet.features.denseblock4,
    base_densenet.features.norm5,
    base_densenet.classifier,
).to(DEVICE).eval()

checkpoint = torch.load(FINETUNED_CNN_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
if checkpoint.get("class_names") != CLASS_NAMES:
    raise ValueError("Checkpoint class order does not match notebook class order.")
if "tail_state_dict" not in checkpoint:
    raise ValueError("Expected selected fine-tuned checkpoint with tail_state_dict.")
if checkpoint.get("metadata", {}).get("model") != "finetuned_densenet201_denseblock4_linear_head":
    raise ValueError("Unexpected checkpoint model metadata.")
tail_model.load_state_dict(checkpoint["tail_state_dict"], strict=True)
frozen_prefix.eval()
tail_model.eval()
print(f"Loaded selected checkpoint from: {FINETUNED_CNN_CHECKPOINT_PATH}")
print(f"Checkpoint best validation epoch: {checkpoint.get('best_epoch')}")
print(f"DenseNet201 feature dimension: {feature_dim}")

## 4. Preload Validation Image Tensors


In [ ]:
def make_loader(frame: pd.DataFrame, batch_size: int = PRELOAD_BATCH_SIZE) -> DataLoader:
    return DataLoader(ImageDataset(frame, preprocess), batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)


@torch.inference_mode()
def preload_validation_images(frame: pd.DataFrame) -> tuple[torch.Tensor, pd.DataFrame]:
    loader = make_loader(frame, batch_size=PRELOAD_BATCH_SIZE)
    image_batches = []
    metadata_rows = []
    for images, labels, feature_input_ids in loader:
        image_batches.append(images)
        for label, feature_input_id in zip(labels.numpy(), feature_input_ids):
            metadata_rows.append({"feature_input_id": feature_input_id, "label_index": int(label)})
    images_cpu = torch.cat(image_batches, dim=0)
    images = images_cpu.to(DEVICE, non_blocking=PIN_MEMORY)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    metadata = pd.DataFrame(metadata_rows)
    return images, metadata


validation_images, validation_image_metadata = preload_validation_images(validation_feature_frame)
preload_summary = pd.DataFrame([
    {
        "validation_images": int(validation_images.shape[0]),
        "unique_validation_fabrics": validation_feature_frame["sample_id"].nunique(),
        "classes": validation_feature_frame["classification_label"].nunique(),
        "tensor_shape": tuple(validation_images.shape),
        "device": str(validation_images.device),
    }
])
preload_summary


## 5. Image-Count Inference Timing

The timing measurement uses preprocessed validation image tensors. Workloads larger than the validation set are created by repeating those tensors in a deterministic cycle. This measures model-only inference scaling after preprocessing, so file reading and PIL image preprocessing are excluded from the timed section.

Each image-count setting runs the selected second-iteration image model with a fixed batch size of 64. The timed path includes the frozen DenseNet201 prefix, fine-tuned DenseNet201 tail, softmax probabilities, and embedding output.


In [ ]:
IMAGE_COUNT_STEPS = [8, 16, 32, 64, 128, 256, 512, 1024, 2048]
SCALABILITY_REPEATS = 5
SCALABILITY_WARMUP_RUNS = 1


def repeated_validation_workload(images: torch.Tensor, image_count: int) -> torch.Tensor:
    if image_count <= 0:
        raise ValueError("image_count must be positive.")
    indices = torch.arange(image_count, device=images.device) % images.shape[0]
    workload = images.index_select(0, indices)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return workload


@torch.inference_mode()
def run_model_only_inference(workload: torch.Tensor) -> None:
    for start in range(0, workload.shape[0], INFERENCE_BATCH_SIZE):
        images = workload[start : start + INFERENCE_BATCH_SIZE]
        prefix_features = frozen_prefix(images)
        logits, embeddings = tail_model(prefix_features)
        _ = torch.softmax(logits, dim=1)
        _ = embeddings


@torch.inference_mode()
def time_image_count_inference(
    images: torch.Tensor,
    image_count: int,
    repeats: int = SCALABILITY_REPEATS,
    warmup_runs: int = SCALABILITY_WARMUP_RUNS,
) -> dict[str, float]:
    workload = repeated_validation_workload(images, image_count)
    frozen_prefix.eval()
    tail_model.eval()

    for _ in range(warmup_runs):
        run_model_only_inference(workload)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed_seconds = []
    for _ in range(repeats):
        if DEVICE.type == "cuda":
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)
            start_event.record()
            run_model_only_inference(workload)
            end_event.record()
            torch.cuda.synchronize()
            elapsed_seconds.append(start_event.elapsed_time(end_event) / 1000.0)
        else:
            start = time.perf_counter()
            run_model_only_inference(workload)
            elapsed_seconds.append(time.perf_counter() - start)

    mean_seconds = float(np.mean(elapsed_seconds))
    std_seconds = float(np.std(elapsed_seconds, ddof=0))
    return {
        "image_count": int(image_count),
        "unique_validation_images": int(min(image_count, images.shape[0])),
        "batch_size": int(INFERENCE_BATCH_SIZE),
        "repeats": int(repeats),
        "mean_seconds": mean_seconds,
        "std_seconds": std_seconds,
        "milliseconds_per_image": mean_seconds * 1000.0 / image_count,
        "images_per_second": image_count / mean_seconds,
    }


scalability_timing = pd.DataFrame([
    time_image_count_inference(validation_images, image_count)
    for image_count in IMAGE_COUNT_STEPS
])
scalability_timing.to_csv(SCALABILITY_TIMING_PATH, index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
x_values = scalability_timing["image_count"].to_numpy()
x_tick_values = np.array([0, 512, 1024, 1536, 2048])
x_tick_labels = [str(value) for value in x_tick_values]

axes[0].plot(x_values, scalability_timing["mean_seconds"], marker="o", color="#2563eb")
axes[0].set_xlim(0, int(x_values.max()) * 1.03)
axes[0].set_xticks(x_tick_values)
axes[0].set_xticklabels(x_tick_labels)
axes[0].set_xlabel("Images processed")
axes[0].set_ylabel("Mean inference time (seconds)")
axes[0].set_title("Total model inference time")
axes[0].grid(alpha=0.25)

axes[1].plot(x_values, scalability_timing["images_per_second"], marker="o", color="#dc2626")
axes[1].set_xlim(0, int(x_values.max()) * 1.03)
axes[1].set_xticks(x_tick_values)
axes[1].set_xticklabels(x_tick_labels)
axes[1].set_xlabel("Images processed")
axes[1].set_ylabel("Images per second")
axes[1].set_title("Model throughput")
axes[1].grid(alpha=0.25)

fig.savefig(SCALABILITY_PLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()
scalability_timing


## 6. Result Note

After execution, this notebook provides a model-only inference scaling table and figure for the selected second CNN iteration. The values describe how runtime changes as the number of images increases after preprocessing, using deterministic repeats of validation images.
